In [32]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, classification_report
)
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

In [33]:
TOKENIZER_NAME = "google/bert_uncased_L-2_H-128_A-2"  # Google's officially maintained
MODEL_NAME = "google/bert_uncased_L-2_H-128_A-2"            # tokenizer-file error on some bert-tiny mirrors.
                                          # Same vocabulary, so nothing breaks.      # the actual model used for embeddings
MAX_LEN = 128
BATCH_SIZE = 32
SAMPLE_SIZE = 3000

In [34]:
print("MODEL_NAME is currently:", MODEL_NAME)

MODEL_NAME is currently: google/bert_uncased_L-2_H-128_A-2


In [35]:
df = pd.read_csv("/content/cleaned_data.csv")
print(f"Loaded {len(df)} rows")

if SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print(f"Using a sample of {len(df)} rows")

X_text = df["review"].astype(str)
y = df["label"].astype(int)

trans_X_train_text, trans_X_test_text, trans_y_train, trans_y_test = train_test_split(
    X_text, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Train: {len(trans_X_train_text)}  Test: {len(trans_X_test_text)}")

Loaded 49582 rows
Using a sample of 3000 rows
Train: 2250  Test: 750


In [36]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
bert_model.eval()
bert_model.to(device)
print(f"Loaded {MODEL_NAME} (tokenizer from {TOKENIZER_NAME}) on {device}")


Running on: cpu


model.safetensors:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded google/bert_uncased_L-2_H-128_A-2 (tokenizer from google/bert_uncased_L-2_H-128_A-2) on cpu


In [37]:
def get_embeddings(texts):
    """One forward pass per review, no gradients -- fast and memory-light."""
    texts = texts.reset_index(drop=True)
    embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding"):
            batch = texts[i:i + BATCH_SIZE].tolist()
            encoded = tokenizer(
                batch, padding=True, truncation=True,
                max_length=MAX_LEN, return_tensors="pt"
            ).to(device)
            output = bert_model(**encoded)
            cls_embeddings = output.last_hidden_state[:, 0, :]  # [CLS] token summary
            embeddings.append(cls_embeddings.cpu().numpy())
    return np.vstack(embeddings)

In [38]:
print("\nExtracting training embeddings...")
trans_X_train_embed = get_embeddings(trans_X_train_text)

print("\nExtracting test embeddings...")
trans_X_test_embed = get_embeddings(trans_X_test_text)


Extracting training embeddings...


Embedding:   0%|          | 0/71 [00:00<?, ?it/s]


Extracting test embeddings...


Embedding:   0%|          | 0/24 [00:00<?, ?it/s]

In [39]:
classifier = Sequential([
    Input(shape=(trans_X_train_embed.shape[1],)),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid"),
])

In [40]:
classifier.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
classifier.summary()

early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

classifier.fit(
    trans_X_train_embed, trans_y_train,
    validation_split=0.2,
    epochs=20,          # EarlyStopping cuts this short once it stops improving
    batch_size=32,
    callbacks=[early_stop],
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,833 (97.00 KB)

 Trainable params: 24,833 (97.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5800 - loss: 0.6921 - val_accuracy: 0.6511 - val_loss: 0.6400
Epoch 2/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6333 - loss: 0.6444 - val_accuracy: 0.6867 - val_loss: 0.6131
Epoch 3/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6544 - loss: 0.6172 - val_accuracy: 0.7178 - val_loss: 0.5949
Epoch 4/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6722 - loss: 0.5950 - val_accuracy: 0.7111 - val_loss: 0.5841
Epoch 5/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6928 - loss: 0.5900 - val_accuracy: 0.7133 - val_loss: 0.5850
Epoch 6/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7122 - loss: 0.5715 - val_accuracy: 0.7333 - val_loss: 0.5687
Epoch 7/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7106 - loss: 0.5598 - val_accuracy: 0.7178 - val_loss: 0.5821
Epoch 8/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7189 - loss: 0.5499 - val_accuracy: 0.7289 - val_loss:

In [41]:
trans_y_pred_prob = classifier.predict(trans_X_test_embed)
trans_y_pred = (trans_y_pred_prob > 0.5).astype(int).ravel()

print("\nAccuracy: ", accuracy_score(trans_y_test, trans_y_pred))
print("Precision:", precision_score(trans_y_test, trans_y_pred))
print("Recall:   ", recall_score(trans_y_test, trans_y_pred))
print("F1 score: ", f1_score(trans_y_test, trans_y_pred))
print()
print(classification_report(trans_y_test, trans_y_pred, target_names=["Negative", "Positive"]))

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 

Accuracy:  0.696
Precision: 0.6835443037974683
Recall:    0.7238605898123325
F1 score:  0.703125

              precision    recall  f1-score   support

    Negative       0.71      0.67      0.69       377
    Positive       0.68      0.72      0.70       373

    accuracy                           0.70       750
   macro avg       0.70      0.70      0.70       750
weighted avg       0.70      0.70      0.70       750



In [42]:
classifier.save("transformer_classifier.keras")
tokenizer.save_pretrained("bert_tokenizer")
bert_model.save_pretrained("bert_base")

print("\nSaved transformer_classifier.keras, bert_tokenizer/, bert_base/")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved transformer_classifier.keras, bert_tokenizer/, bert_base/


In [43]:
!zip -rq transformer_artifacts.zip transformer_classifier.keras bert_tokenizer bert_base
from google.colab import files
files.download("transformer_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>